# Installiere ultralytics Bibliothek

In [ ]:
!pip install ultralytics

## Import Libraries

In [1]:
import os
from zipfile import ZipFile
import ultralytics
from ultralytics import YOLO

In [2]:
path = os.getcwd()
print(path)

d:\Studium\5. Semester\PA2\Glass-Defect-Detection-Evaluating-Object-Detection-Models\Coding\Model_Code\YOLO


## Load in the data and unzip it

In [ ]:
# Create the directory if it doesn't exist
extract_dir = "data"
os.makedirs(extract_dir, exist_ok=True)

with ZipFile("Glass Defect Detection.v3i.yolov11.zip", "r") as data_set:
    data_set.extractall(extract_dir)

## Load the Model


In [4]:
model = YOLO("yolo11n.pt")

## Create configurations


In [8]:
DATA_YAML = path +"/Glass Defect DEtection.v3i.yolov11/data.yaml"
MODEL_PATH = "yolo11n.pt"
data_set_dir = "data"

## Fine-Tuning with Hyperparameters

In [ ]:
results = model.train(
    data = DATA_YAML,
    epochs = 100,
    batch = 16,
    lr0 = 0.0001,
    val = True
)

Testing the Model

In [6]:
model = YOLO("best.pt")

In [9]:
metrics = model.val(
    data = DATA_YAML,
    split  = "test",
    batch = 16
    )

print(metrics.box.map)

Ultralytics 8.3.225  Python-3.12.10 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1070, 8192MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 37.444.8 MB/s, size: 15.0 KB)
val: Scanning D:\Studium\5. Semester\PA2\Glass-Defect-Detection-Evaluating-Object-Detection-Models\Coding\Model_Code\YOLO\Glass Defect Detection.v3i.yolov11\test\labels... 173 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 173/173 1.1Kit/s 0.2s<0.2s
val: New cache created: D:\Studium\5. Semester\PA2\Glass-Defect-Detection-Evaluating-Object-Detection-Models\Coding\Model_Code\YOLO\Glass Defect Detection.v3i.yolov11\test\labels.cache
WARNING Box and segment counts should be equal, but got len(segments) = 24, len(boxes) = 370. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 4.7i

In [12]:
metrics.results_dict

{'metrics/precision(B)': 0.8690170239277916,
 'metrics/recall(B)': 0.7990901674101504,
 'metrics/mAP50(B)': 0.8396457530558581,
 'metrics/mAP50-95(B)': 0.5897596903283091,
 'fitness': 0.5897596903283091}

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

cm_raw_data = np.array([
    # P:defect  | T:defect (145), T:glass (59), T:background (0 - angenommen)
    [145, 0, 59], 
    # P:glass   | T:defect (52), T:glass (167), T:background (7)
    [0, 167, 7],
    # P:background| T:defect (0 - angenommen), T:glass (6), T:background (0 - angenommen)
    [52, 6, 0]
])


# Transponieren der Matrix: (Predicted x True) --> (True x Predicted)
cm_final_data = cm_raw_data.T

# Standard-Labels mit Umbenennung von 'background'
standard_labels = ['defect', 'glass', 'no defect']


# --- 2. Plot der korrigierten 3x3 Matrix ---
plt.figure(figsize=(9, 8))
sns.heatmap(
    cm_final_data, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    cbar=True,
    xticklabels=standard_labels,
    yticklabels=standard_labels
)

plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix: YOLO, Roboflow')
plt.show()

# Print der finalen Matrix
print("\nKonsolidierte 3x3 Matrix (True=Reihe vs. Predicted=Spalte):\n")
df = pd.DataFrame(cm_final_data, index=standard_labels, columns=standard_labels)
print(df)